### Prepare development environment

use Python 3.11.x for CUDA Version: 12.6
> py -3.11 -m venv torch-env

> torch-env\Scripts\activate

Install libraries:

> pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

> pip install fast-langdetect joblib matplotlib nltk numpy pandas scipy sentence-transformers scikit-learn umap-learn xgboost hdbscan

In [1]:
from fast_langdetect import detect
import hdbscan
import joblib
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd
from pathlib import Path
import re
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sentence_transformers import SentenceTransformer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import hamming_loss, accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC
import torch
import umap # umap-learn
import unicodedata
from xgboost import XGBClassifier

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

d:\diogo\Desktop\MsEx_IA_apl\LLMs\aulas\torch-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\diogo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\diogo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\diogo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\diogo\AppData\Roaming\nltk_data...


Using device: cuda


[nltk_data]   Package wordnet is already up-to-date!


# Read dataset

In [2]:
def read_dataset(file_path):
    df = pd.read_csv(file_path, usecols=["book_desc", "genres"])
    return df

df = read_dataset("books_dataset.csv")
print(df.shape)
print(df.dtypes)

(54301, 2)
book_desc    str
genres       str
dtype: object


# Analyze data

### Analyze description texts

In [3]:
print(df["book_desc"].head(10))
print(df["book_desc"].describe())

0    Winning will make you famous. Losing means cer...
1    There is a door at the end of a silent corrido...
2    The unforgettable novel of a childhood in a sl...
3    «È cosa ormai risaputa che a uno scapolo in po...
4    About three things I was absolutely positive.F...
5    Trying to make sense of the horrors of World W...
6    Journeys to the end of the world, fantastic cr...
7    مزرعة الحيوانات هي رائعة جورج أورويل الخالدة.....
8    Gone with the Wind is a novel written by Marga...
9    لجزء الثالث من ملحمة جيه أر أر تولكين الرائعة ...
Name: book_desc, dtype: str
count                                                 52970
unique                                                51781
top       هذه هي طبعة "دار الفكر - بيروت" وهي آخر طبعة ع...
freq                                                     38
Name: book_desc, dtype: object


conclusions from this code block:
- only found 52970 descriptions over 54301 rows, this means there are some NA (missing) values
- identified multiple languages and writing systems, as for example Abjad (Arabic)
- unique values (51781) is lower than all texts found (52970), this means there are some duplicated entries

In [4]:
repeated = df.groupby(["book_desc"]).size().reset_index(name="count")
repeated = repeated[repeated["count"] > 1]

print(repeated)

                                               book_desc  count
238    \r\n\r\n\r\nBased on the true story of a forgo...      2
557    \r\n\r\nWar has erupted in the Banished Lands ...      2
926     Until now, he's existed only in her dreams -b...      2
956     milk and honey  is a collection of poetry and...      2
1135   "As Gregor Samsa awoke one morning from uneasy...      2
...                                                  ...    ...
51258  “Did you ever hear the Tragedy of Darth Plague...      2
51304  “Her knees were weak and if not for that marbl...      2
51461  “Our Dragon doesn’t eat the girls he takes, no...      2
51640  „Als ich zum ersten Mal in deine Augen sah, wu...      2
51649  „Dievų miškas“ — memuarinis lietuvių rašytojo ...      2

[926 rows x 2 columns]


conclusions from this code block:
- found entries with carriage returns and line feeds, this don't add useful semantic information, may harm the tokenization process or produce slightly different embeddings for the same sentence.

# Preprocess training data

### Cleanup and normalize dataset

In [5]:
def clean_text(text):
    # if the text is NaN (missing value), return it as is to avoid processing
    if pd.isna(text):
        return text
    text = str(text)
    # remove carriage returns and line feeds
    text = re.sub(r"(\\[rn]|[\r\n])+", " ", text)
    # replace multiple whitespace characters with a single space and trim leading/trailing whitespace
    text = re.sub(r"\s+", " ", text).strip()
    # if the resulting text is empty after cleaning, return None to indicate missing value
    if text == "":
        return None
    # normalize to unicode to ensure consistent representation of characters
    text = unicodedata.normalize("NFKC", text)
    return text

def clean_dataset(df):
    df["book_desc_clean"] = df["book_desc"].apply(clean_text)
    # drop null values from descriptions and genres, as they cannot be used for training the model
    df = df.dropna(subset=["book_desc_clean", "genres"])
    # drop duplicated descriptions
    df = df.drop_duplicates(subset="book_desc_clean", keep="last")
    # reset index after dropping rows to avoid issues with indexing later on
    df = df.reset_index(drop=True)
    return df

print(df.shape)
df = clean_dataset(df)
print(df.shape)

(54301, 2)
(49002, 3)


Pre-processing performed:
- For each description:
    - if there is a value, removed all the carriage returns, line feeds and additional whitespaces,
    - if the text is not empty then normalize text fixing any Unicode inconsistencies, helpful specially when dealing with multiple languages and writing systems
- Remove all rows with missing values on description or genres
- Remove duplicated descriptions, keeping the last of the duplicated, assuming the last one would be the most updated record (just for learning purposes)

## Vectorize description texts

#### Language issue


descriptions in multiple languages and we dont have a column identifying the language to filter

In [6]:
lang_confidence_threshold = 0.7

def detect_language(text):
    result = detect(str(text), model='lite', k=1)
    return result[0]["lang"], result[0]["score"]

def add_language_info(df):
    df[["lang", "lang_confidence"]] = df["book_desc_clean"].apply(lambda x: pd.Series(detect_language(x)))
    df["high_conf"] = df["lang_confidence"] > lang_confidence_threshold
    return df

df = add_language_info(df)
languages_stats = df.groupby("lang").agg(
    high_confidence=("high_conf", "sum"),
    total=("lang", "size")
)
print(languages_stats.to_string())

      high_confidence  total
lang                        
af                  0      1
ar                680    695
arz                 1      2
azb                 0      1
bg                 71     77
bn                 26     26
bs                  0      1
ca                  8     12
ceb                 0      3
cs                105    108
da                 41     68
de                574    641
el                 94     95
en              39383  42674
eo                  0      2
es                828    891
et                 16     21
fa                126    135
fi                 90     94
fr                555    594
fy                  0      1
gl                  1      3
hi                  5      6
hr                  0     32
hu                 26     29
hy                  1      1
id                201    337
is                  3      3
it                391    433
ja                 54     55
ka                 39     39
ko                  5      6
la            

Conclusions from this code block:
- Besides english we dont have a large amount of descriptions for each language
- For english, we have 37517 records with language confidence higher than 80%

So we have two options to vectorize the descriptions:
1) use TF-IDF vectorizer, that requires some additional NLP strategies to reduce noise on the data while keeping the context, like stopwords removal and lemmatization, but this approach is only viable for the the books that we are almost certain that are in english, as it is the only language that we have acceptable amount of data, 39383 records.
2) Use a pretrained multilingual embeddings model (as Sentence-BERT) to generate a fixed-size vector that supports multiple languages and preserve semantics, as it is almost language agnostic, we can use all the descriptions, 49002


### Approach 1 - Use tf-idf together with more preprocessing to vectorize descriptions

In [7]:
vectorizer_file = "./generated/book_desc_vectorizer_tfidf.joblib"

def lemmatize_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return " ".join(tokens)

def vectorize_descriptions_with_tfidf(descriptions):
    print("Lemmatizing descriptions...")
    lemmas = []
    for desc in descriptions:
        lemmas.append(lemmatize_text(str(desc)))

    if Path(vectorizer_file).exists():
        print("Loading precomputed vectorizer from file...")
        vectorizer = joblib.load(vectorizer_file)
    else:
        vectorizer = TfidfVectorizer(max_features=5000)

    if(not hasattr(vectorizer, "vocabulary_")):
        print("Fitting vectorizer to descriptions...")
        vectorizer.fit(lemmas)
        joblib.dump(vectorizer, vectorizer_file)

    X = vectorizer.transform(lemmas).toarray()

    return X

df_english = df[(df["lang"] == "en") & df["high_conf"]]
X = vectorize_descriptions_with_tfidf(df_english["book_desc_clean"])


Lemmatizing descriptions...
Loading precomputed vectorizer from file...


### Approach 2 - Use transformer to vectorize descriptions

In [8]:
embeddings_file = "./generated/book_desc_embeddings_sbert.npy"

def vectorize_descriptions_with_sbert(descriptions, use_file=True):
    if use_file and Path(embeddings_file).exists():
        print("Loading precomputed embeddings from file...")
        return np.load(embeddings_file)
    
    transformer = SentenceTransformer('distiluse-base-multilingual-cased-v2')
    X = transformer.encode(
        list(descriptions),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        device=device
    )

    if use_file:    
        np.save(embeddings_file, X)
        
    return X

# descriptions = df['book_desc_clean']
# vectorize_descriptions_with_sbert(descriptions)

## Multi-label binarization of genres

#### Multi-labels issue

In [12]:
few_genres = df["genres"].apply(lambda x: str(x).split("|")).explode().value_counts()

print("Total: ", few_genres.size, "\n")
print("Genres with less than 20 occurrences:", few_genres[few_genres < 20], f"\n{few_genres[few_genres < 20].size/few_genres.size*100:.2f}%\n")

Total:  864 

Genres with less than 20 occurrences: genres
Young Adult Romance    19
Futuristic             19
Role Playing Games     19
Americana              18
Nigeria                18
                       ..
Disability Studies      1
Peak Oil                1
Health Care             1
Read For College        1
Anthropomorphic         1
Name: count, Length: 430, dtype: int64 
49.77%



#### Create embeddings for labels

In [16]:
def create_label_embeddings(genres_list):
    # extract unique genres
    all_genres = list(set([genre for book_genres in genres_list for genre in book_genres]))

    global device

    model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight and fast
    embeddings = model.encode(
        all_genres,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        device=device
    )
    
    # https://www.datacamp.com/pt/tutorial/understanding-umap-guide-to-dimensionality-reduction
    umap_embeddings = umap.UMAP(n_neighbors=20, min_dist=0.0, metric='cosine').fit_transform(embeddings)
    
    return all_genres, umap_embeddings

genres_list = df["genres"].apply(lambda x: str(x).split("|"))
all_genres, embeddings = create_label_embeddings(genres_list)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2710.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 14/14 [00:02<00:00,  5.13it/s]


#### Apply HDBScan clustering

goals:
1) isolate noise (outliers)
2) avoid enforcement of cluster number

In [18]:
def cluster_labels_with_hdbscan(embeddings, all_genres):

    cluster_labels = hdbscan.HDBSCAN(min_cluster_size=10, metric='euclidean').fit_predict(embeddings)

    # Map each genre to its HDBSCAN cluster
    genre_to_label = {genre: label for genre, label in zip(all_genres, cluster_labels)}

    # Apply again HDBScan on the outliers with lower cluster size to refine sparser groups, avoiding too much genres under "Other" group
    outlier_idx = np.where(cluster_labels == -1)[0]
    outlier_genres = [all_genres[i] for i in outlier_idx]

    if len(outlier_genres) > 0:
        sub_clusterer = hdbscan.HDBSCAN(
            min_cluster_size=2,  # smaller to capture niche clusters
            metric='euclidean'
        )

        # get UMAP embeddings for outlier genres
        outlier_embeddings = embeddings[outlier_idx]
        # HDBSCAN on outliers
        sub_labels = sub_clusterer.fit_predict(outlier_embeddings)

        # Offset sub-cluster labels so they don't overlap with first pass
        max_label = cluster_labels.max()
        sub_labels = [s + max_label + 1 if s != -1 else -1 for s in sub_labels]

        # Update main genre_to_hdbscan mapping
        for genre, sub_label in zip(outlier_genres, sub_labels):
            genre_to_label[genre] = sub_label
            
    return genre_to_label

# genre_to_label = cluster_labels_with_hdbscan(embeddings, all_genres)

#### Apply Hierarchical Clustering

In [19]:
def cluster_labels_with_hierarchical_clustering(embeddings, all_genres, print_dendrogram=False):
    # Compute hierarchical linkage
    Z = linkage(embeddings, method='ward')  # 'ward' minimizes variance within clusters

    if print_dendrogram:
        # Plot dendrogram, useful for exploring structure
        plt.figure(figsize=(15, 5))
        dendrogram(Z, labels=all_genres, leaf_rotation=90, leaf_font_size=8)
        plt.show()

    # Cut tree to get clusters at desired level
    cluster_labels = fcluster(Z, 50, criterion='maxclust')

    # Map genres to clusters
    genre_to_label = {genre: label for genre, label in zip(all_genres, cluster_labels)}
    return genre_to_label

# genre_to_label = cluster_labels_with_hierarchical_clustering(embeddings, all_genres, print_dendrogram=True)

#### Analyse Clusters

In [20]:
# Calculate the importance of each cluster based on the number of books and genre occurrences that belong to that cluster
def get_cluster_names(genres_list, genre_to_label, print_clusters_info=False):
    label_occurrences = {}
    for genre, label in genre_to_label.items():
        if label not in label_occurrences:
            label_occurrences[label] = {
                genre: genres_list.apply(lambda genres: genre in genres).sum()
            }
        else:
            label_occurrences[label][genre] = genres_list.apply(lambda genres: genre in genres).sum()

    clusters_names = {label: max(genres, key=lambda k: genres[k]) for label, genres in label_occurrences.items()}
    clusters_names[-1] = "Other"

    if print_clusters_info:
        print(f"Total Clusters: {label_occurrences.__len__()}\n")
        for label, genres in sorted(label_occurrences.items(), key=lambda x: sum(x[1].values()), reverse=True):
            print(f"Cluster {label:02d} \"{clusters_names[label]}\": \n occurrences: {sum(genres.values())}\n genres: {', '.join(genres.keys())}\n")

    return clusters_names

# clusters_names = get_cluster_names(df, genre_to_label, print_clusters_info=True)

In [21]:
def print_cluster_by_label(label, genre_to_label, clusters_names):
    print(f"\nCheck Cluster {label} \"{clusters_names[label]}\":")
    for genre, cluster_label in genre_to_label.items():
        if cluster_label == label:
            print(f"  {genre}")

# print_cluster_by_label(-1, genre_to_label, clusters_names)
# print_cluster_by_label(genre_to_label["Bdsm"], genre_to_label, clusters_names)

#### Group genres and binarize 

In [43]:
def binarize_genres(genres_list, clustering_method, filename_suffix):
    genre_to_label_filename = f"./generated/genre_to_label_{filename_suffix}.joblib"
    
    if filename_suffix and Path(genre_to_label_filename).exists():
        print(f"Using {genre_to_label_filename}...")
        genre_to_label = joblib.load(genre_to_label_filename)
    else:
        all_genres, embeddings = create_label_embeddings(genres_list)
        genre_to_label = clustering_method(embeddings, all_genres)
        joblib.dump(genre_to_label, genre_to_label_filename)

    clusters_names = get_cluster_names(genres_list, genre_to_label)
    labelled_genres = genres_list.apply(lambda book_genres: list(set([clusters_names[genre_to_label[genre]] for genre in book_genres if genre in genre_to_label])))

    mlb_filename = f"./generated/mlb_{filename_suffix}.joblib"
    if filename_suffix and Path(mlb_filename).exists():
        print(f"Using {mlb_filename}...")
        mlb = joblib.load(mlb_filename)
    else:
        mlb = MultiLabelBinarizer()
        mlb.fit(labelled_genres)
        joblib.dump(mlb, mlb_filename)
        
    Y = mlb.transform(labelled_genres)
    
    return Y, mlb

genres_list = df["genres"].apply(lambda x: str(x).split("|"))
Y, mlb = binarize_genres(genres_list, cluster_labels_with_hdbscan, "TFIDF_HDBSCAN")

print(Y.shape)
print(mlb.classes_)



Using ./generated/genre_to_label_TFIDF_HDBSCAN.joblib...
Using ./generated/mlb_TFIDF_HDBSCAN.joblib...
(54301, 68)
['Action' 'Adventure' 'Africa' 'American' 'Angels' 'Animals'
 'Anthropology' 'Apocalyptic' 'Architecture' 'Art History' 'Audiobook'
 'Bande Dessinée' 'Brain' 'Chick Lit' 'Christian' 'Christian Fiction'
 'Classics' 'Comics' 'Contemporary' 'Crime' 'Cultural' 'Dragons' 'Drama'
 'Erotica' 'Espionage' 'Fantasy' 'Feminism' 'Fiction' 'Food and Drink'
 'France' 'Harlequin' 'Historical' 'Holiday' 'Humor' 'Inspirational'
 'International Relations' 'Lgbt' 'Linguistics' 'Literature' 'Love'
 'M M Romance' 'Marriage' 'Middle Grade' 'Military History' 'Mystery'
 'Nature' 'Nonfiction' 'Novels' 'Other' 'Poetry' 'Politics' 'Psychology'
 'Reference' 'Roman' 'Romance' 'Science' 'Science Fiction Fantasy'
 'Self Help' 'Sequential Art' 'Sex Work' 'Shapeshifters'
 'Social Movements' 'Space' 'Sports' 'War' 'Westerns' 'World War II'
 'Young Adult']


d:\diogo\Desktop\MsEx_IA_apl\LLMs\aulas\torch-env\Lib\site-packages\sklearn\preprocessing\_label.py:1007: UserWarning: unknown class(es) ['Gardening', 'Nobel Prize', 'Philosophy'] will be ignored
  warnings.warn(


# Train model

General use case to train model using an algorithm sent by parameter, saving in a file for future use and returning the intended data for testing the model

In [23]:
def train_with(algorithm, X, Y, filename_suffix):
    # Split the dataset into training and testing sets (80% train, 20% test)
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    filename = f"./generated/genre_classifier_{filename_suffix}.joblib"
    # If a filename is provided and the file exists, load the model from the file. Otherwise, train a new model and save it to the file.
    if filename_suffix and Path(filename).exists():
        print(f"Using {filename}...")
        model = joblib.load(filename)
        return model, X_test, Y_test

    # Initialize the OneVsRestClassifier with the specified algorithm
    model = OneVsRestClassifier(algorithm)
    # Train the model on the training data
    model.fit(X_train, Y_train)
    # Save the trained model to a file for future use
    joblib.dump(model, filename)

    return model, X_test, Y_test


In [ ]:
Path('./generated').mkdir(parents=True, exist_ok=True)

df = read_dataset("books_dataset.csv")
df = clean_dataset(df)

df = add_language_info(df)
english_df = df[(df["lang"] == "en") & df["high_conf"]]
X = vectorize_descriptions_with_tfidf(english_df["book_desc_clean"])

genres_list = english_df["genres"].apply(lambda x: str(x).split("|"))
Y, mlb = binarize_genres(genres_list, cluster_labels_with_hdbscan, "TFIDF_HDBSCAN")


Total Clusters: 68

Cluster 12 "Fiction": 
 occurrences: 33910
 genres: Science Fiction, Speculative Fiction, Lesbian Fiction, Womens Fiction, Young Adult Historical Fiction, Light Novel, Literary Fiction, Amish Fiction, Flash Fiction, Historical Fiction, Gay Fiction, Weird Fiction, Fiction, American Fiction, Fan Fiction, Animal Fiction, Graphic Non Fiction, Realistic Fiction, Adult Fiction, Military Fiction, Bizarro Fiction, Female Authors

Cluster 13 "Mystery": 
 occurrences: 17788
 genres: Thriller, Mystery, Gothic Horror, Cozy Mystery, Ghosts, Mystery Thriller, Buffy The Vampire Slayer, Suspense, Werewolves, Murder Mystery, Psychological Thriller, Paranormal Mystery, Legal Thriller, Gothic, Horror, Supernatural, Southern Gothic, Spy Thriller, Vampires, Golden Age Mystery, Paranormal

Cluster 14 "Fantasy": 
 occurrences: 17470
 genres: Fantasy, Folk Tales, Dark, Surreal, Dark Fantasy, High Fantasy, Fae, Historical Fantasy, Epic Fantasy, Cinderella, Fairy Tales, Mermaids, Storytime, 

['./generated/genre_to_label_english.joblib']

### Train with Logistic regression

In [ ]:
model, X_test, Y_test = train_with(LogisticRegression(max_iter=2000), X, Y)

joblib.dump(model, f"./generated/genre_classifier_TFIDF_HDBSCAN_LogisticRegression.joblib")

['./generated/genre_classifier_tfidf_hdbscan_logisticregression.joblib']

### Train with SVM
In theory has better accuracy between all non-NN algorithm, it maximizes margin between classes and handles well high dimensionality as the case of vectorized texts.

In [ ]:
model, X_test, Y_test = train_with(CalibratedClassifierCV(LinearSVC()), X, Y)

joblib.dump(model, f"./generated/genre_classifier_TFIDF_HDBSCAN_svm.joblib")

['./generated/genre_classifier_tfidf_hdbscan_svm.joblib']

### Training with XGBoost
In theory, using a non-linear classifier that captures co-occurrence patterns, as for example, fantasy + adventure often occur together

In [ ]:
model, X_test, Y_test = train_with(
    XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        device=device
    ), X, Y
)

joblib.dump(model, f"./generated/model_TFIDF_HDBSCAN_XGBoost.joblib")

['./generated/genre_classifier_tfidf_hdbscan_xgboost.joblib']

# Test model

#### Run test data and analyse metrics

In [ ]:
model = joblib.load("./generated/genre_classifier_TFIDF_HDBSCAN_LogisticRegression.joblib")
Y_probs = model.predict_proba(X_test)

thresholds = (0.5, 0.4, 0.3, 0.2, 0.1)
values = [["Hamming Loss"], ["Accuracy"], ["Precision (micro)"], ["Recall (micro)"], ["F1 Score (micro)"]]
for threshold in thresholds:
    genre_pred = (Y_probs >= threshold).astype(int)

    hloss = hamming_loss(Y_test, genre_pred)
    acc = accuracy_score(Y_test, genre_pred)
    precision_micro = precision_score(Y_test, genre_pred, average='micro')
    recall_micro = recall_score(Y_test, genre_pred, average='micro')
    f1_micro = f1_score(Y_test, genre_pred, average='micro')

    values[0].append(f"{hloss:.4f}")
    values[1].append(f"{acc:.4f}")
    values[2].append(f"{precision_micro:.4f}")
    values[3].append(f"{recall_micro:.4f}")
    values[4].append(f"{f1_micro:.4f}")

results_df = pd.DataFrame(values, columns=["Metric"] + [str(th) for th in thresholds])
print(results_df)

              Metric     0.5     0.4     0.3     0.2     0.1
0       Hamming Loss  0.0381  0.0382  0.0411  0.0501  0.0839
1           Accuracy  0.0853  0.0868  0.0705  0.0383  0.0025
2  Precision (micro)  0.7722  0.7175  0.6444  0.5461  0.3874
3     Recall (micro)  0.4687  0.5433  0.6185  0.7040  0.8156
4   F1 Score (micro)  0.5834  0.6183  0.6312  0.6150  0.5253


conclusions from this 
- Hamming Loss = 0.0047: very low hamming loss is good, it means most individual labels are correctly predicted, but can be misleading as we have a lot of genres (864) and most of them are 0 por book
- Accuracy = 0.0277: very low accuracy is NOT good, but is usual in multi-label problems, because getting every label correct, knowing every book may have multiple genres, would be over exceptional
- Micro Precision = 0.7372: high micro precision is good, means a lot of predicted labels are actually correct, but may be, again, due to the fact we have a lot os labels and usually a book only have 2/3 genres, so all the other 800 would be 0
- Micro Recall = 0.3245: low micro recall es NOT good, this means it misses a lot of true genres, that is actually the goal
- Micro F1 Score = 04507: harmonic mean of micro-precision and micro-recall across all labels, will reflect the evolution of the model
- High precision + low recall + low F1 = the model is cautious, it predicts few labels, those it predicts are usually correct (high precision), but it misses many true genres (low recall)
- We can lower the threshold of the labels, but then there is a trade-off, precision decreases, recall increases
- We could even search the best threshold to apply for each gender, optimizing that way the f1 score

#### Use model with custom descriptions

# Run full pipeline

### Use predefined thresholds

In [ ]:
df = read_dataset("books_dataset.csv")
print("Dataset loaded successfully.")

df = clean_dataset(df)
print("Dataset cleaned successfully.")

df = add_language_info(df)
print("Language information added successfully.")

vectorizers = (
    ("SBERT", vectorize_descriptions_with_sbert, df),
    ("TFIDF", vectorize_descriptions_with_tfidf, df[(df["lang"] == "en") & df["high_conf"]])
)

clusterings = (
    ("HDBSCAN", cluster_labels_with_hdbscan),
    ("HierarchicalClustering", cluster_labels_with_hierarchical_clustering),
)

classifiers = (
    ("LogisticRegression", LogisticRegression(max_iter=2000)),
    ("SVM", CalibratedClassifierCV(LinearSVC())),
    ("XGBoost", XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, device=device)),
)

thresholds = (0.5, 0.4, 0.3, 0.2, 0.1)

with open("./generated/results.csv", mode="w") as results_file:
    for vectorizer_name, vectorizer_func, vectorizer_df in vectorizers:
        print(f"\nVectorizer: {vectorizer_name}")
        X = vectorizer_func(vectorizer_df["book_desc_clean"])
        
        genres_list = vectorizer_df["genres"].apply(lambda x: str(x).split("|"))
        
        for clustering_name, clustering_func in clusterings:
            print(f"Clustering: {clustering_name}")

            binarizer_filename_suffix = f"{vectorizer_name}_{clustering_name}"
        
            Y, mlb = binarize_genres(genres_list, clustering_func, binarizer_filename_suffix)

            for classifier_name, classifier_func in classifiers:
                print(f"Classifier: {classifier_name}")
                
                model_filename_suffix = f"{binarizer_filename_suffix}_{classifier_name}"
                
                model, X_test, Y_test = train_with(classifier_func, X, Y, model_filename_suffix)

                Y_probs = model.predict_proba(X_test)

                values = [["Hamming Loss"], ["Accuracy"], ["Precision"], ["Recall"], ["F1 Score"]]
                for value in values:
                    value.append(f"{vectorizer_name} + {clustering_name} + {classifier_name}")

                for threshold in thresholds:
                    genre_pred = (Y_probs >= threshold).astype(int)

                    hloss = hamming_loss(Y_test, genre_pred)
                    acc = accuracy_score(Y_test, genre_pred)
                    precision_micro = precision_score(Y_test, genre_pred, average="micro")
                    recall_micro = recall_score(Y_test, genre_pred, average="micro")
                    f1_micro = f1_score(Y_test, genre_pred, average="micro")

                    values[0].append(f"{hloss:.4f}")
                    values[1].append(f"{acc:.4f}")
                    values[2].append(f"{precision_micro:.4f}")
                    values[3].append(f"{recall_micro:.4f}")
                    values[4].append(f"{f1_micro:.4f}")

                
                
                for value in values:
                    results_file.write(",".join(value)+"\n")
    results_file.close()



Dataset loaded successfully.
Dataset cleaned successfully.
Language information added successfully.

Vectorizer: SBERT
Loading precomputed embeddings from file...
Clustering: HierarchicalClustering
Using ./generated/genre_to_label_SBERT_HierarchicalClustering.joblib...
Using ./generated/mlb_SBERT_HierarchicalClustering.joblib...
Classifier: XGBoost


d:\diogo\Desktop\MsEx_IA_apl\LLMs\aulas\torch-env\Lib\site-packages\xgboost\core.py:751: UserWarning: [18:26:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Clustering: HDBSCAN
Using ./generated/genre_to_label_SBERT_HDBSCAN.joblib...
Using ./generated/mlb_SBERT_HDBSCAN.joblib...
Classifier: XGBoost

Vectorizer: TFIDF
Lemmatizing descriptions...
Loading precomputed vectorizer from file...
Clustering: HierarchicalClustering
Using ./generated/genre_to_label_TFIDF_HierarchicalClustering.joblib...
Using ./generated/mlb_TFIDF_HierarchicalClustering.joblib...
Classifier: XGBoost
Clustering: HDBSCAN
Using ./generated/genre_to_label_TFIDF_HDBSCAN.joblib...
Using ./generated/mlb_TFIDF_HDBSCAN.joblib...
Classifier: XGBoost
Using ./generated/genre_classifier_TFIDF_HDBSCAN_XGBoost.joblib...


### Search best F1 score (micro) threshold for each label

In [ ]:
# Instead of brute-force thresholds, compute optimal thresholds, faster and more precise
from sklearn.metrics import precision_recall_curve

df = read_dataset("books_dataset.csv")
print("Dataset loaded successfully.")

df = clean_dataset(df)
print("Dataset cleaned successfully.")

df = add_language_info(df)
print("Language information added successfully.")

vectorizers = (
    ("SBERT", vectorize_descriptions_with_sbert, df),
    ("TFIDF", vectorize_descriptions_with_tfidf, df[(df["lang"] == "en") & df["high_conf"]])
)

clusterings = (
    ("HDBSCAN", cluster_labels_with_hdbscan),
    ("HierarchicalClustering", cluster_labels_with_hierarchical_clustering),
)

classifiers = (
    ("LogisticRegression", LogisticRegression(max_iter=2000)),
    ("SVM", CalibratedClassifierCV(LinearSVC())),
    ("XGBoost", XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, device=device)),
)

with open("./generated/results_optimized.csv", mode="w") as results_file:
    for vectorizer_name, vectorizer_func, vectorizer_df in vectorizers:
        print(f"\nVectorizer: {vectorizer_name}")
        X = vectorizer_func(vectorizer_df["book_desc_clean"])
        
        genres_list = vectorizer_df["genres"].apply(lambda x: str(x).split("|"))
        
        for clustering_name, clustering_func in clusterings:
            print(f"Clustering: {clustering_name}")

            binarizer_filename_suffix = f"{vectorizer_name}_{clustering_name}"
        
            Y, mlb = binarize_genres(genres_list, clustering_func, binarizer_filename_suffix)

            for classifier_name, classifier_func in classifiers:
                print(f"Classifier: {classifier_name}")
                
                model_filename_suffix = f"{binarizer_filename_suffix}_{classifier_name}"
                
                model, X_test, Y_test = train_with(classifier_func, X, Y, model_filename_suffix)

                Y_probs = model.predict_proba(X_test)

                n_labels = Y_probs.shape[1]
                best_thresholds = np.zeros(n_labels)

                for i in range(n_labels):
                    y_true_label = Y_test[:, i]
                    y_prob_label = Y_probs[:, i]

                    precision, recall, thresh = precision_recall_curve(y_true_label, y_prob_label)

                    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
                    best_idx = np.argmax(f1_scores)

                    best_thresholds[i] = thresh[best_idx]

                np.save(f"./generated/thresholds_{model_filename_suffix}.npy", best_thresholds)

                algorithm_desc = f"{vectorizer_name} + {clustering_name} + {classifier_name}"
                    
                genre_pred = (Y_probs >= best_thresholds).astype(int)

                hloss = hamming_loss(Y_test, genre_pred)
                acc = accuracy_score(Y_test, genre_pred)
                precision_micro = precision_score(Y_test, genre_pred, average="micro")
                recall_micro = recall_score(Y_test, genre_pred, average="micro")
                f1_micro = f1_score(Y_test, genre_pred, average="micro")

                results_file.write(f"Hamming Loss,{algorithm_desc},{hloss:.4f}\n")
                results_file.write(f"Accuracy,{algorithm_desc},{acc:.4f}\n")
                results_file.write(f"Precision,{algorithm_desc},{precision_micro:.4f}\n")
                results_file.write(f"Recall,{algorithm_desc},{recall_micro:.4f}\n")
                results_file.write(f"F1 Score,{algorithm_desc},{f1_micro:.4f}\n")

    results_file.close()

Dataset loaded successfully.
Dataset cleaned successfully.
Language information added successfully.

Vectorizer: SBERT
Loading precomputed embeddings from file...
Clustering: HDBSCAN
Using ./generated/genre_to_label_SBERT_HDBSCAN.joblib...
Using ./generated/mlb_SBERT_HDBSCAN.joblib...
Classifier: LogisticRegression
Using ./generated/genre_classifier_SBERT_HDBSCAN_LogisticRegression.joblib...
Classifier: SVM
Using ./generated/genre_classifier_SBERT_HDBSCAN_SVM.joblib...
Classifier: XGBoost
Using ./generated/genre_classifier_SBERT_HDBSCAN_XGBoost.joblib...
Clustering: HierarchicalClustering
Using ./generated/genre_to_label_SBERT_HierarchicalClustering.joblib...
Using ./generated/mlb_SBERT_HierarchicalClustering.joblib...
Classifier: LogisticRegression
Using ./generated/genre_classifier_SBERT_HierarchicalClustering_LogisticRegression.joblib...
Classifier: SVM
Using ./generated/genre_classifier_SBERT_HierarchicalClustering_SVM.joblib...
Classifier: XGBoost
Using ./generated/genre_classifie

# Use models

In [ ]:
threshold = 0.3

# vectorizer = "TFIDF"
vectorizer = "SBERT"

texts = (
    "London, 1888. Beneath the fog-filled streets and flickering gas lamps, the city whispers with rumors of a series of strange disappearances. Each vanished person was last seen near the same narrow alley in Whitechapel—and each left behind an identical object: a small, intricately crafted pocket watch that always stops at exactly 3:17 a.m.",
    "In a distant future where advanced technology has unlocked portals to forgotten realms, humanity discovers a hidden universe filled with ancient magic and mythical creatures. When a young astrophysicist accidentally activates a long-dormant gateway orbiting Earth, she unleashes a mysterious force that begins merging science and sorcery.",
    "The Parker family just wanted a peaceful weekend getaway in the countryside. Instead, they end up renting an old mansion that the locals refuse to talk about. According to the listing, the house is “full of character.” What it doesn’t mention is that the character includes a grumpy ghost, a mischievous shadow creature, and a cursed doll that really hates loud children.",
    "Lena never believed in fate. As a determined art student trying to balance college, part-time jobs, and her dreams of becoming a painter, she’s convinced life is all about hard work—not destiny. Then she literally collides with Noah. Noah is a quiet but thoughtful engineering student who prefers books and late-night coffee over crowded parties. When Lena accidentally spills an entire cup of coffee over his carefully organized notes in a campus café, their first meeting is anything but romantic.",
    "For as long as they can remember, five friends have shared one strange thing in common: absolutely terrible luck.\nThere’s Jake, who once slipped on a banana peel… in a parking lot… where there were no bananas.\nMaya, whose phone always dies exactly when something important happens.\nLuis, who somehow sets off alarms in stores even when he doesn’t buy anything.\nSophie, whose GPS constantly sends the group to the wrong places.\nAnd Ben, who has broken six chairs in his life just by sitting down.\nDespite their ridiculous streak of misfortune, they decide to finally do something exciting together: a weekend road trip to attend a huge festival.",
    "Num reino onde a magia é proibida e as estrelas são vistas como presságios de desgraça, Elara esconde um segredo que pode mudar tudo: ela consegue ouvir o sussurro do céu.\nQuando conhece Kael, um misterioso viajante com um passado envolto em sombras, o destino de ambos entrelaça-se de forma inesperada. À medida que forças antigas despertam e uma guerra se aproxima, Elara e Kael são obrigados a confiar um no outro para sobreviver."
)
# # texts_vectors = vectorize_descriptions_with_tfidf(texts)
texts_vectors = vectorize_descriptions_with_sbert(texts, use_file=False)
    
# clustering = "HierarchicalClustering"
clustering = "HDBSCAN"

# classifier = "LogisticRegression"
# classifier = "SVM"
classifier = "XGBoost"


model = joblib.load(f"./generated/genre_classifier_{vectorizer}_{clustering}_{classifier}.joblib")
mlb = joblib.load(f"./generated/mlb_{vectorizer}_{clustering}.joblib")
thresholds = np.load(f"./generated/thresholds_{vectorizer}_{clustering}_{classifier}.npy")
print("Model loaded successfully.")


# Predict probabilities
genre_pred = model.predict_proba(texts_vectors)

for i, probs in enumerate(genre_pred):
    print(f"Book {i+1} genre probabilities:")
    for j, genre in enumerate(mlb.classes_): 
        if probs[j] >= thresholds[j]: 
            print(f" {genre}: {probs[j]:.4f}")

Model loaded successfully.
Book 1 genre probabilities:
 Classics: 0.3145
 Cultural: 0.2724
 Fantasy: 0.7484
 Fiction: 0.8367
 Literature: 0.4992
 Steampunk: 0.0299
 Travel: 0.0208
Book 2 genre probabilities:
 Fantasy: 0.9790
 Fiction: 0.7609
 Science Fiction: 0.9614
 Space: 0.0109
 Steampunk: 0.0799
 Young Adult: 0.3762
Book 3 genre probabilities:
 Fantasy: 0.8684
 Fiction: 0.6829
 Gothic: 0.0040
 Young Adult: 0.4744
Book 4 genre probabilities:
 Fantasy: 0.5037
 Fiction: 0.7037
 Romance: 0.7908
 Sequential Art: 0.3101
 Young Adult: 0.4565
Book 5 genre probabilities:
 Fantasy: 0.5933
 Fiction: 0.7070
 Young Adult: 0.5566
Book 6 genre probabilities:
 Fantasy: 0.9902
 Young Adult: 0.5205
